# 06 — Files, JSON, SQLite & the CLI
### `pathlib`, `json`, `sqlite3`, `argparse`, `os.environ`, `datetime`

This notebook covers the standard-library modules that give the project its
I/O: reading KB files off disk, talking to a real (if tiny) database,
serializing results, and turning `pipeline.py` into a command-line tool via
`main.py`.

## 6.1 `pathlib.Path` — modern file paths

`Path` objects represent filesystem paths and support `/` for joining,
plus methods like `.glob()` to find files matching a pattern —
`database.py` and `retrieval.py` both use this instead of manually building
path strings.

In [ ]:
from pathlib import Path

kb_dir = Path(".") / "kb"
print(kb_dir)
print(kb_dir.exists())

md_files = sorted(kb_dir.glob("*.md"))
for f in md_files:
    print(f.name, "-", f.stat().st_size, "bytes")


Compare to `retrieval.py`:

```python
KB_DIR = Path(__file__).parent / "kb"
...
for md_file in sorted(kb_dir.glob("*.md")):
    content = md_file.read_text(encoding="utf-8")
```

`Path(__file__).parent` means "the folder this Python file itself lives
in" — that's how the project finds `kb/` correctly no matter what
directory you *run* the script from.

## 6.2 `sqlite3` — a real (if tiny) database

Python's standard library ships a full SQL database with zero extra
installs. `database.py` uses three ideas worth knowing: connecting,
parameterized queries (never string-formatting SQL directly), and
`row_factory` to get dict-like rows back instead of plain tuples.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")   # a temporary, in-RAM database -- gone when the connection closes
conn.execute("CREATE TABLE demo (id TEXT, name TEXT)")
conn.execute("INSERT INTO demo VALUES (?, ?)", ("C1", "Ananya"))  # ? placeholders -- SAFE from SQL injection
conn.commit()

row = conn.execute("SELECT * FROM demo WHERE id = ?", ("C1",)).fetchone()
print(row)             # a plain tuple by default: ('C1', 'Ananya')
print(row[1])          # accessing by position only -- easy to get wrong as columns grow

conn.close()


**Why the `?` placeholder matters:** `conn.execute("... WHERE id = ?", ("C1",))`
is safe even if `"C1"` came from untrusted user input, because SQLite
handles the substitution itself. Writing
`f"... WHERE id = '{customer_id}'"` instead would be a SQL injection
vulnerability — if `customer_id` ever contained something like
`"'; DROP TABLE customers; --"`, that string would execute as SQL. Always
use `?` placeholders, never f-strings, for values in a query.

In [ ]:
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row   # <-- this line is the whole trick
conn.execute("CREATE TABLE demo (id TEXT, name TEXT)")
conn.execute("INSERT INTO demo VALUES (?, ?)", ("C1", "Ananya"))
conn.commit()

row = conn.execute("SELECT * FROM demo WHERE id = ?", ("C1",)).fetchone()
print(row["name"])       # now accessible by column name
print(dict(row))         # and convertible straight to a plain dict

conn.close()


This is exactly `database.get_customer()`'s trick:

```python
def get_customer(customer_id: str) -> dict | None:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    row = conn.execute("SELECT * FROM customers WHERE customer_id = ?", (customer_id,)).fetchone()
    conn.close()
    return dict(row) if row else None
```

`row_factory = sqlite3.Row` plus `dict(row)` is how a SQL row becomes the
plain dict every other function in the project expects.

## 6.3 `executemany` — inserting many rows at once

`database.py`'s seed data uses `executemany` to insert all customers (or
all orders) in one call instead of looping with individual `execute` calls.

In [ ]:
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE demo (id TEXT, name TEXT)")

rows = [("C1", "Ananya"), ("C2", "Rohan"), ("C3", "Priya")]
conn.executemany("INSERT INTO demo VALUES (?, ?)", rows)
conn.commit()

for row in conn.execute("SELECT * FROM demo"):
    print(row)

conn.close()


## 6.4 `json` — serializing dicts to text and back

`pipeline.py` and `main.py` use `json.dumps()` to turn a case report (a
nested dict) into a printable/saveable string.

In [ ]:
import json

report = {
    "ticket_id": "T001",
    "resolution_path": "auto_resolved",
    "classification": {"issue_type": "order_status", "confidence": 0.9},
}

as_text = json.dumps(report, indent=2)
print(as_text)

back_to_dict = json.loads(as_text)
print(back_to_dict["classification"]["issue_type"])


### The `default=str` trick for non-JSON-serializable values

`datetime` objects (and Pydantic model instances, and Enum members in some
configurations) aren't natively JSON-serializable — `json.dumps` will raise
a `TypeError` on them unless you tell it how to convert anything it doesn't
recognize.

In [ ]:
from datetime import datetime, timezone

report_with_time = {"ticket_id": "T001", "handled_at": datetime.now(timezone.utc)}

try:
    json.dumps(report_with_time)
except TypeError as e:
    print(f"Without default=str: {e}")

print()
print("With default=str:")
print(json.dumps(report_with_time, default=str))   # falls back to str(value) for anything it can't handle natively


This is exactly why `main.py` calls
`json.dumps(report, indent=2, default=str)` — `report["handled_at"]` is a
real `datetime` object, and `default=str` tells `json.dumps` to convert
anything it doesn't know how to serialize by just calling `str()` on it.

## 6.5 `os.environ` — reading configuration from the environment

`llm_client.py` decides mock-vs-live mode and which model to use by reading
environment variables, with a fallback default if they're not set.

In [ ]:
import os

mode = os.environ.get("SUPPORTPILOT_MODE", "mock")     # "mock" is the default if the env var isn't set
print(mode)

os.environ["SUPPORTPILOT_MODE"] = "live"   # simulate setting it before the module is imported
print(os.environ.get("SUPPORTPILOT_MODE", "mock"))


This pattern — read a setting from the environment with a sensible default
— is how the whole project switches between offline testing and hitting a
real API without changing a single line of code, only an environment
variable set before you run it:

```bash
export SUPPORTPILOT_MODE=live
export ANTHROPIC_API_KEY=sk-...
python3 main.py --ticket "..."
```

## 6.6 `argparse` — turning a script into a CLI tool

`main.py` uses `argparse` to accept command-line flags like `--init-db`,
`--test-suite`, and `--ticket "..."`, instead of hardcoding one behavior
per script.

In [ ]:
import argparse

parser = argparse.ArgumentParser(description="Demo CLI")
parser.add_argument("--name", type=str, default="World")
parser.add_argument("--shout", action="store_true")   # a flag with no value -- True if present, False if not

# argparse normally reads sys.argv; we pass a list directly here to simulate
# command-line input inside a notebook.
args = parser.parse_args(["--name", "Ananya", "--shout"])
greeting = f"Hello, {args.name}!"
print(greeting.upper() if args.shout else greeting)

args2 = parser.parse_args([])   # no flags given -- defaults apply
print(args2.name, args2.shout)


Compare to `main.py`:

```python
parser.add_argument("--init-db", action="store_true")
parser.add_argument("--ticket", type=str)
parser.add_argument("--customer", type=str, default=None)
...
if args.init_db:
    database.init_db(force=True)
    ...
```

Same two patterns: `action="store_true"` for on/off flags, `type=str` (or
`type=int`, etc.) plus an optional `default=` for flags that take a value.

## Exercise

1. Write a function `save_report(report: dict, path: str) -> None` that
   writes a dict to a JSON file, handling `datetime` values safely.
   (Hint: `json.dump(report, open(path, "w"), default=str)`.)
2. Write a small `argparse` CLI with one flag, `--seed`, that takes an
   integer and prints `f"Seeding with value {args.seed}"` — give it a
   default of `42`.
3. Open `database.py` and find every place `?` placeholders are used in a
   query. For each one, write down what would go wrong (concretely, not
   just "it's unsafe") if that query were built with an f-string instead.